# 🧠 Fase 1 — O Modelo Entende Português?
## Embeddings e Similaridade Semântica com NorBERTo

---
**Roteiro-Desafio-NLP · Fatec DSM 6º Semestre · 2026**  
**Grupo:** ___________________________  **Data:** ___/___/______

---

### 🎯 Objetivo desta fase
Nesta fase você vai aprender o que são **embeddings contextuais** — as representações numéricas que modelos de linguagem como o NorBERTo criam para cada palavra e frase.  
Você vai usar o NorBERTo *sem treinar nada*, apenas para extrair vetores e medir **similaridade semântica** entre frases em português.

### 📚 O que você vai aprender
- O que é tokenização e como funciona no NorBERTo
- O que são embeddings e para que servem
- Como calcular similaridade cosseno entre frases
- Como visualizar embeddings com t-SNE

### 🔗 Recursos utilizados
- Modelo: [`Itau-Unibanco/NorBERTo-base`](https://huggingface.co/Itau-Unibanco/NorBERTo-base)
- Dataset: [`Itau-Unibanco/FAQ_BACEN`](https://huggingface.co/datasets/Itau-Unibanco/FAQ_BACEN)
- Artigo: *NorBERTo: A ModernBERT Model Trained for Portuguese* (PROPOR 2026)

---


## 📦 Passo 1 — Instalação das bibliotecas
Vamos instalar tudo que precisamos. Execute esta célula e aguarde.

In [ ]:
# Instala as bibliotecas necessárias para esta fase
!pip install transformers datasets torch scikit-learn matplotlib seaborn -q

print("✅ Bibliotecas instaladas com sucesso!")


## 🔍 Passo 2 — Carregando o NorBERTo-base
O NorBERTo é um modelo **encoder-only** baseado na arquitetura ModernBERT, treinado com 331 bilhões de tokens em português pelo Itaú-Unibanco.

Aqui vamos carregar o modelo e seu tokenizador diretamente do Hugging Face.


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME = "Itau-Unibanco/NorBERTo-base"

print(f"⏳ Carregando tokenizador e modelo: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # Modo de inferência — sem atualização de pesos

print(f"✅ Modelo carregado!")
print(f"   Parâmetros: ~150 milhões")
print(f"   Vocabulário: {tokenizer.vocab_size:,} tokens")


## ✂️ Passo 3 — Tokenização: como o modelo lê o texto
Antes de processar qualquer frase, o modelo precisa convertê-la em **tokens** (pedaços de texto com IDs numéricos).

Vamos ver como o NorBERTo tokeniza frases em português:


In [ ]:
# Vamos tokenizar algumas frases e ver o resultado
frases_exemplo = [
    "O banco central fixou a taxa de juros.",
    "A inflação subiu acima da meta.",
    "Gosto muito de comer pizza."
]

print("="*60)
for frase in frases_exemplo:
    tokens = tokenizer.tokenize(frase)
    ids = tokenizer.encode(frase)
    print(f"\n📝 Frase: '{frase}'")
    print(f"   Tokens ({len(tokens)}): {tokens}")
    print(f"   IDs:    {ids}")
print("="*60)

print("\n💡 Observe que palavras raras são divididas em subpalavras!")
print("   Ex: 'inflação' pode virar ['inflação'] ou ['infla', '##ção']")


## 🧮 Passo 4 — Gerando Embeddings
Embedding é um vetor de números que representa o **significado** de uma frase no espaço matemático do modelo.
Frases semanticamente parecidas ficam próximas nesse espaço.

Vamos criar uma função para extrair embeddings usando o NorBERTo:


In [ ]:
import torch
import numpy as np

def get_embedding(texto, tokenizer, model):
    """
    Gera o embedding de uma frase usando o NorBERTo.
    
    Estratégia: pegamos a representação do token [CLS],
    que o BERT usa para codificar o significado global da frase.
    """
    # Tokeniza e converte para tensores PyTorch
    inputs = tokenizer(texto, return_tensors="pt", 
                       truncation=True, max_length=512, padding=True)
    
    # Inferência sem calcular gradientes (mais rápido)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Pega a representação do token [CLS] (índice 0 da última camada)
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    # Converte para numpy e normaliza (para usar similaridade cosseno)
    embedding = cls_embedding.squeeze().numpy()
    embedding = embedding / np.linalg.norm(embedding)
    return embedding

# Teste
emb = get_embedding("Taxa de juros brasileira", tokenizer, model)
print(f"✅ Embedding gerado com sucesso!")
print(f"   Dimensão: {emb.shape[0]} dimensões")
print(f"   Primeiros 5 valores: {emb[:5].round(4)}")


## 📐 Passo 5 — Similaridade Cosseno
A **similaridade cosseno** mede o ângulo entre dois vetores.
- Valor próximo de **1.0** → frases muito parecidas semanticamente
- Valor próximo de **0.0** → frases sem relação
- Valor próximo de **-1.0** → frases opostas

Vamos testar com frases do domínio financeiro e comparar com frases aleatórias:


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Conjuntos de frases para comparar
frases = [
    # Financeiro
    "O Banco Central do Brasil controla a inflação.",
    "A taxa Selic é o principal instrumento de política monetária.",
    "O IPCA mede a inflação oficial do Brasil.",
    "O PIB cresceu 2% no último trimestre.",
    # Cotidiano (não financeiro)
    "Vou ao mercado comprar frutas amanhã.",
    "O filme foi muito emocionante e empolgante.",
    "Meu cachorro gosta de brincar no parque.",
]

print("⏳ Gerando embeddings para todas as frases...")
embeddings = np.array([get_embedding(f, tokenizer, model) for f in frases])
print("✅ Embeddings gerados!\n")

# Calcula matriz de similaridade
sim_matrix = cosine_similarity(embeddings)

# Exibe as similaridades mais relevantes
print("="*60)
print("📊 SIMILARIDADES ENTRE FRASES SELECIONADAS")
print("="*60)
pares = [
    (0, 1, "Financeiro × Financeiro (relacionados)"),
    (0, 2, "Financeiro × Financeiro (relacionados)"),
    (0, 4, "Financeiro × Cotidiano (não relacionados)"),
    (4, 5, "Cotidiano × Cotidiano"),
]
for i, j, desc in pares:
    sim = sim_matrix[i][j]
    barra = "█" * int(sim * 20)
    print(f"\n{desc}")
    print(f"  A: '{frases[i][:50]}...' " if len(frases[i])>50 else f"  A: '{frases[i]}'")
    print(f"  B: '{frases[j][:50]}...' " if len(frases[j])>50 else f"  B: '{frases[j]}'")
    print(f"  Similaridade: {sim:.4f}  {barra}")
print("="*60)


## 📊 Passo 6 — Carregando o Dataset FAQ_BACEN
Vamos usar um dataset real do Itaú-Unibanco no Hugging Face: perguntas e respostas do FAQ do Banco Central do Brasil.


In [ ]:
from datasets import load_dataset

print("⏳ Carregando dataset FAQ_BACEN...")
dataset = load_dataset("Itau-Unibanco/FAQ_BACEN", split="train")
print(f"✅ Dataset carregado: {len(dataset)} registros")
print(f"   Colunas: {dataset.column_names}")
print()

# Visualiza os primeiros exemplos
for i, ex in enumerate(dataset.select(range(5))):
    print(f"Exemplo {i+1}:")
    for col in dataset.column_names:
        val = str(ex[col])[:100]
        print(f"  {col}: {val}{'...' if len(str(ex[col]))>100 else ''}")
    print()


## 🔎 Passo 7 — Busca Semântica com o FAQ_BACEN
Agora vamos usar os embeddings do NorBERTo para fazer **busca semântica**: dado uma pergunta do usuário, encontrar a entrada mais similar no FAQ.


In [ ]:
# Seleciona uma amostra do dataset para o experimento
amostra = dataset.select(range(50))

# Identifica a coluna de texto (pergunta)
col_pergunta = dataset.column_names[0]

print(f"⏳ Gerando embeddings para {len(amostra)} perguntas do FAQ...")
textos_faq = [ex[col_pergunta] for ex in amostra]
embeddings_faq = np.array([get_embedding(t, tokenizer, model) for t in textos_faq])
print("✅ Pronto!\n")

def busca_semantica(consulta, textos, embeddings, top_k=3):
    """Encontra as frases mais similares à consulta."""
    emb_consulta = get_embedding(consulta, tokenizer, model).reshape(1, -1)
    similaridades = cosine_similarity(emb_consulta, embeddings)[0]
    indices_top = similaridades.argsort()[::-1][:top_k]
    return [(textos[i], similaridades[i]) for i in indices_top]

# Teste de busca semântica
consultas = [
    "Como funciona o Pix?",
    "O que é taxa de juros?",
    "Como reclamar de um banco?"
]

for consulta in consultas:
    print(f"🔍 Consulta: '{consulta}'")
    resultados = busca_semantica(consulta, textos_faq, embeddings_faq)
    for rank, (texto, sim) in enumerate(resultados, 1):
        print(f"  {rank}. [{sim:.3f}] {texto[:80]}{'...' if len(texto)>80 else ''}")
    print()


## 🗺️ Passo 8 — Visualização com t-SNE
O t-SNE reduz os embeddings de 768 dimensões para 2D, permitindo visualizar como o modelo organiza os conceitos no espaço semântico.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.manifold import TSNE

# Conjunto de frases com categorias conhecidas
frases_categorizadas = {
    "Financeiro": [
        "Taxa Selic e política monetária",
        "IPCA e inflação no Brasil",
        "Pix e pagamentos instantâneos",
        "Crédito e empréstimos bancários",
        "Câmbio e moeda estrangeira",
    ],
    "Saúde": [
        "Sintomas de gripe e resfriado",
        "Vacinas e imunização",
        "Pressão arterial e hipertensão",
        "Diabetes e controle glicêmico",
        "Exercício físico e saúde",
    ],
    "Tecnologia": [
        "Inteligência artificial e machine learning",
        "Redes neurais e deep learning",
        "Computação em nuvem e servidores",
        "Programação Python e algoritmos",
        "Segurança cibernética e senhas",
    ],
}

todas_frases, rotulos, categorias = [], [], list(frases_categorizadas.keys())
for cat, frases in frases_categorizadas.items():
    for f in frases:
        todas_frases.append(f)
        rotulos.append(cat)

print("⏳ Gerando embeddings e aplicando t-SNE...")
embs = np.array([get_embedding(f, tokenizer, model) for f in todas_frases])
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
coords = tsne.fit_transform(embs)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
colors = {"Financeiro": "#0D9488", "Saúde": "#F59E0B", "Tecnologia": "#6366F1"}
markers = {"Financeiro": "o", "Saúde": "s", "Tecnologia": "^"}

for i, (frase, rotulo) in enumerate(zip(todas_frases, rotulos)):
    ax.scatter(coords[i, 0], coords[i, 1],
               c=colors[rotulo], marker=markers[rotulo], s=120, zorder=3,
               label=rotulo if i == list(rotulos).index(rotulo) else "")
    ax.annotate(frase[:30]+"…" if len(frase)>30 else frase,
                (coords[i, 0], coords[i, 1]),
                fontsize=7, ha='center', va='bottom',
                xytext=(0, 6), textcoords='offset points')

handles = [plt.Line2D([0],[0], marker=markers[c], color='w',
           markerfacecolor=colors[c], markersize=10, label=c) for c in categorias]
ax.legend(handles=handles, title="Categoria", loc="upper right")
ax.set_title("Visualização t-SNE dos Embeddings do NorBERTo\n(frases por domínio semântico)", 
             fontsize=13, fontweight='bold')
ax.set_xlabel("Dimensão 1 (t-SNE)")
ax.set_ylabel("Dimensão 2 (t-SNE)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("tsne_fase1.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n💡 Observe como frases do mesmo domínio ficam agrupadas!")


## 📝 Avaliação — Fase 1
Responda as questões abaixo com base no que você explorou nesta fase.  
Anote as respostas no espaço indicado.

---

**Q1.** O NorBERTo-base possui aproximadamente quantos parâmetros?  
( ) 15 milhões  ( ) 150 milhões  ( ) 1,5 bilhão  ( ) 15 bilhões

**Q2.** Qual é o tamanho do vocabulário do NorBERTo?  
( ) ~10.000 tokens  ( ) ~30.000 tokens  ( ) ~50.000 tokens  ( ) ~100.000 tokens

**Q3.** O que representa o token [CLS] em modelos BERT?  
( ) O primeiro caractere da frase  
( ) Uma representação global do significado da frase  
( ) O token de separação entre frases  
( ) Um marcador de fim de sequência

**Q4.** Qual é a dimensão do vetor de embedding gerado pelo NorBERTo-base?  
( ) 128  ( ) 256  ( ) 512  ( ) 768

**Q5.** O que indica um valor de similaridade cosseno próximo de 1.0?  
( ) As frases são opostas semanticamente  
( ) As frases são semanticamente muito parecidas  
( ) O modelo errou o cálculo  
( ) As frases têm o mesmo número de palavras

**Q6.** Por que normalizamos os embeddings antes de calcular a similaridade cosseno?  
( ) Para reduzir o tempo de processamento  
( ) Para que a métrica dependa apenas da direção, não do tamanho dos vetores  
( ) Para deixar todos os valores entre 0 e 1  
( ) Porque o PyTorch exige normalização

**Q7.** O que o t-SNE faz com os embeddings?  
( ) Treina o modelo NorBERTo  
( ) Aumenta a dimensão dos vetores para melhor visualização  
( ) Reduz a dimensionalidade para 2D preservando estrutura de vizinhança  
( ) Calcula a similaridade entre todas as frases

**Q8.** No experimento de busca semântica, o NorBERTo encontra resultados similares baseado em:  
( ) Palavras exatas em comum  
( ) Significado semântico representado pelos embeddings  
( ) Número de caracteres da frase  
( ) Ordem alfabética das palavras

**Q9.** O corpus Aurora-PT, usado para treinar o NorBERTo, contém aproximadamente quantos tokens?  
( ) 2,68 bilhões  ( ) 15 bilhões  ( ) 100 bilhões  ( ) 331 bilhões

**Q10.** Qual arquitetura base o NorBERTo utiliza?  
( ) GPT-2  ( ) RoBERTa  ( ) ModernBERT  ( ) T5

---
*Entrega: anote as respostas e discuta em grupo antes de prosseguir para a Fase 2.*
